# ResNet18 Baseline — Colab Version
### NeuroReach | Mohina Rustamova
**Goal:** Train ResNet18 on OASIS, evaluate on OASIS and ADNI, save weights permanently to Google Drive.

In [1]:
# ============================================================
# Cell 1 — Mount Google Drive + Setup Kaggle API
# ============================================================
# We save all outputs to Google Drive so weights persist
# even after Colab session ends.

from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/NeuroReach', exist_ok=True)
print('Google Drive mounted.')
print('Output folder: /content/drive/MyDrive/NeuroReach')

Mounted at /content/drive
Google Drive mounted.
Output folder: /content/drive/MyDrive/NeuroReach


In [2]:
# ============================================================
# Cell 2 — Setup Kaggle API and Download Datasets
# ============================================================
# Upload your kaggle.json when prompted

from google.colab import files
print('Upload your kaggle.json file:')
uploaded = files.upload()  # Upload kaggle.json here

# Setup Kaggle credentials
os.makedirs('/root/.kaggle', exist_ok=True)
os.rename('/content/kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('Kaggle API configured.')

Upload your kaggle.json file:


Saving kaggle.json to kaggle.json
Kaggle API configured.


In [3]:
# ============================================================
# Cell 3 — Download OASIS Dataset from Kaggle
# ============================================================
# This downloads OASIS directly from Kaggle into Colab
# Takes ~5 minutes depending on connection speed

!kaggle datasets download -d ninadaithal/imagesoasis -p /content/oasis --unzip
print('OASIS downloaded.')

# Find the data path
for root, dirs, files_list in os.walk('/content/oasis'):
    for d in dirs:
        print(os.path.join(root, d))
    break

Dataset URL: https://www.kaggle.com/datasets/ninadaithal/imagesoasis
License(s): apache-2.0
100% 1.23G/1.23G [00:17<00:00, 76.4MB/s]

OASIS downloaded.
/content/oasis/Data


In [4]:
# ============================================================
# Cell 4 — Download ADNI Dataset from Kaggle
# ============================================================

!kaggle datasets download -d mohinarustamova/neuroreach-adni-coral -p /content/adni --unzip
print('ADNI downloaded.')

# Find the data path
count = 0
for root, dirs, files_list in os.walk('/content/adni'):
    for f in files_list[:1]:
        print(os.path.join(root, f))
        count += 1
    if count >= 5:
        break

Dataset URL: https://www.kaggle.com/datasets/mohinarustamova/neuroreach-adni-coral
License(s): unknown
100% 538M/538M [00:07<00:00, 74.9MB/s]

ADNI downloaded.
/content/adni/neuroreach_adni_coral_6_24_2026.csv
/content/adni/neuroreach_adni_coral/ADNI/002_S_0413/MPRAGE_SENS/2006-05-19_16_17_47.0/I15800/ADNI_002_S_0413_MR____________MPRAGE_SENS_br_raw_20060519180337163_68_S14782_I15800.dcm
/content/adni/neuroreach_adni_coral/ADNI/002_S_0413/MPRAGE_REPE/2006-05-19_16_29_07.0/I15799/ADNI_002_S_0413_MR____________MPRAGE_REPE_br_raw_20060519180837844_103_S14781_I15799.dcm
/content/adni/neuroreach_adni_coral/ADNI/002_S_0729/MPRAGE/2006-08-02_07_13_17.0/I20087/ADNI_002_S_0729_MR_MPRAGE_br_raw_20070223050508369_100_S17534_I20087.dcm
/content/adni/neuroreach_adni_coral/ADNI/002_S_0729/MPRAGE/2006-08-02_07_02_00.0/I20088/ADNI_002_S_0729_MR_MPRAGE_br_raw_20070223050407881_92_S17535_I20088.dcm


In [6]:
# ============================================================
# Cell 5 — Imports + GPU Check
# ============================================================

!pip install pydicom -q
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler, random_split, Dataset
from torchvision import transforms, models, datasets
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import pydicom
import pandas as pd
from pathlib import Path
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 38.6 MB/s eta 0:00:00
Device: cuda
GPU: Tesla T4


In [7]:
# ============================================================
# Cell 6 — OASIS DataLoaders
# ============================================================
# Update DATA_DIR after checking Cell 3 output

DATA_DIR = '/content/oasis/Data'  # update if needed
IMG_SIZE = 224
BATCH_SIZE = 32

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_test_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(root=DATA_DIR)
class_names = full_dataset.classes
print('Classes:', class_names)
print('Total images:', len(full_dataset))

total = len(full_dataset)
train_size = int(0.7 * total)
val_size = int(0.15 * total)
test_size = total - train_size - val_size

train_set, val_set, test_set = random_split(
    full_dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

train_set.dataset.transform = train_transforms
val_set.dataset.transform   = val_test_transforms
test_set.dataset.transform  = val_test_transforms

targets = [full_dataset.targets[i] for i in train_set.indices]
class_counts = np.bincount(targets)
class_weights = 1.0 / class_counts
sample_weights = [class_weights[t] for t in targets]
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, sampler=sampler)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False)

print(f'Train: {train_size} | Val: {val_size} | Test: {test_size}')

Classes: ['Mild Dementia', 'Moderate Dementia', 'Non Demented', 'Very mild Dementia']
Total images: 86437
Train: 60505 | Val: 12965 | Test: 12967


In [8]:
# ============================================================
# Cell 7 — Build ResNet18 Model
# ============================================================

def get_model(num_classes=4):
    model = models.resnet18(pretrained=True)
    for param in model.parameters():
        param.requires_grad = True
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model.to(device)

model = get_model()
print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 124MB/s]


Trainable params: 11,178,564


In [9]:
# ============================================================
# Cell 8 — Train ResNet18 on OASIS
# ============================================================

def train_model(model, train_loader, val_loader, criterion, optimizer, epochs=5):
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    for epoch in range(epochs):
        model.train()
        train_loss, train_correct, train_total = 0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            preds = outputs.argmax(dim=1)
            train_correct += (preds == labels).sum().item()
            train_total += labels.size(0)

        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                preds = outputs.argmax(dim=1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

        train_acc = train_correct / train_total * 100
        val_acc   = val_correct / val_total * 100
        history['train_loss'].append(train_loss / len(train_loader))
        history['val_loss'].append(val_loss / len(val_loader))
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        print(f'Epoch [{epoch+1}/{epochs}] Train Loss: {train_loss/len(train_loader):.4f} | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%')
    return history

history = train_model(model, train_loader, val_loader, criterion, optimizer, epochs=5)

Epoch [1/5] Train Loss: 0.0857 | Train Acc: 96.88% | Val Acc: 98.74%
Epoch [2/5] Train Loss: 0.0179 | Train Acc: 99.41% | Val Acc: 98.96%
Epoch [3/5] Train Loss: 0.0114 | Train Acc: 99.64% | Val Acc: 99.64%
Epoch [4/5] Train Loss: 0.0090 | Train Acc: 99.70% | Val Acc: 99.97%
Epoch [5/5] Train Loss: 0.0091 | Train Acc: 99.70% | Val Acc: 99.93%


In [10]:
# ============================================================
# Cell 9 — Evaluate ResNet18 on OASIS Test Set
# ============================================================

def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return all_preds, all_labels

preds, labels = evaluate(model, test_loader)
print('=== ResNet18 on OASIS (Test Set) ===')
acc = np.mean(np.array(preds) == np.array(labels)) * 100
print(f'Accuracy: {acc:.2f}%')
print(classification_report(labels, preds, target_names=class_names))

=== ResNet18 on OASIS (Test Set) ===
Accuracy: 99.89%
                    precision    recall  f1-score   support

     Mild Dementia       0.99      1.00      0.99       742
 Moderate Dementia       1.00      1.00      1.00        82
      Non Demented       1.00      1.00      1.00     10061
Very mild Dementia       1.00      1.00      1.00      2082

          accuracy                           1.00     12967
         macro avg       1.00      1.00      1.00     12967
      weighted avg       1.00      1.00      1.00     12967



In [11]:
# ============================================================
# Cell 10 — Save ResNet18 Weights to Google Drive
# ============================================================

SAVE_PATH = '/content/drive/MyDrive/NeuroReach/resnet18_baseline.pth'
torch.save(model.state_dict(), SAVE_PATH)
print(f'ResNet18 weights saved to: {SAVE_PATH}')

ResNet18 weights saved to: /content/drive/MyDrive/NeuroReach/resnet18_baseline.pth


In [12]:
# ============================================================
# Cell 11 — Load ADNI DICOMs + Build Labeled Dataset
# ============================================================

ADNI_ROOT = '/content/adni/neuroreach_adni_coral/ADNI'
CSV_PATH  = '/content/adni/neuroreach_adni_coral_6_24_2026.csv'

# Load labels
df = pd.read_csv(CSV_PATH)
GROUP_MAP = {'CN': 2, 'MCI': 3, 'AD': 0}
subject_labels = df[['Subject', 'Group']].drop_duplicates('Subject')
subject_to_label = {row['Subject']: GROUP_MAP[row['Group']] for _, row in subject_labels.iterrows()}
print(f'Subjects with labels: {len(subject_to_label)}')
print(f'Distribution: {pd.Series(subject_to_label.values()).value_counts().to_dict()}')

# Collect DICOM files
adni_files = []
subject_ids = []
for dcm_path in Path(ADNI_ROOT).rglob('*.dcm'):
    subject_id = dcm_path.parts[-5]
    adni_files.append(str(dcm_path))
    subject_ids.append(subject_id)

subject_to_files = defaultdict(list)
for path, subj in zip(adni_files, subject_ids):
    subject_to_files[subj].append(path)
for subj in subject_to_files:
    subject_to_files[subj].sort()

print(f'Total DICOM files: {len(adni_files)}')
print(f'Unique subjects: {len(subject_to_files)}')

# ADNI Labeled Dataset
adni_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class ADNILabeledDataset(Dataset):
    def __init__(self, subject_to_files, subject_to_label, slices_per_subject=10, transform=None):
        self.transform = transform
        self.samples = []
        for subj, files in subject_to_files.items():
            if subj not in subject_to_label:
                continue
            label = subject_to_label[subj]
            n = len(files)
            start = int(0.20 * n)
            end   = int(0.80 * n)
            usable = files[start:end]
            indices = np.linspace(0, len(usable) - 1, slices_per_subject, dtype=int)
            for i in indices:
                self.samples.append((usable[i], label))
        print(f'Total labeled samples: {len(self.samples)}')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        dcm = pydicom.dcmread(path)
        img = dcm.pixel_array.astype(np.float32)
        img = (img - img.min()) / (img.max() - img.min() + 1e-8)
        img = np.stack([img, img, img], axis=0)
        img = torch.tensor(img, dtype=torch.float32)
        if self.transform:
            img = self.transform(img)
        return img, label

adni_dataset = ADNILabeledDataset(subject_to_files, subject_to_label, slices_per_subject=10, transform=adni_transforms)
adni_loader  = DataLoader(adni_dataset, batch_size=32, shuffle=False)

Subjects with labels: 27
Distribution: {3: 11, 2: 9, 0: 7}
Total DICOM files: 8794
Unique subjects: 27
Total labeled samples: 270


In [15]:
# ============================================================
# Cell 12 — Evaluate ResNet18 on ADNI (Zero-Shot)
# ============================================================

preds_adni, labels_adni = evaluate(model, adni_loader)

print('=== ResNet18 on ADNI (Zero-Shot, No Domain Adaptation) ===')
acc_adni = np.mean(np.array(preds_adni) == np.array(labels_adni)) * 100
print(f'Accuracy: {acc_adni:.2f}%')
print(classification_report(labels_adni, preds_adni,
      target_names=['CN (Normal)', 'MCI', 'AD (Alzheimers)'],
      zero_division=0))

=== ResNet18 on ADNI (Zero-Shot, No Domain Adaptation) ===
Accuracy: 30.37%
                 precision    recall  f1-score   support

    CN (Normal)       0.38      0.11      0.18        70
            MCI       0.30      0.60      0.40        90
AD (Alzheimers)       0.28      0.18      0.22       110

       accuracy                           0.30       270
      macro avg       0.32      0.30      0.27       270
   weighted avg       0.31      0.30      0.27       270



In [16]:
# ============================================================
# Cell 13 — Final Summary Table
# ============================================================

print('=' * 55)
print('NEUROREACH MODULE 1 — RESULTS SUMMARY')
print('=' * 55)
print(f'{"Model":<25} {"Dataset":<10} {"Accuracy"}')
print('-' * 55)
print(f'{"ResNet18 (CNN)":<25} {"OASIS":<10} {acc:.2f}%')
print(f'{"ResNet18 (CNN)":<25} {"ADNI":<10} {acc_adni:.2f}%  <- domain gap')
print(f'{"ViT":<25} {"OASIS":<10} 99.98%')
print(f'{"ViT + CORAL":<25} {"ADNI":<10} 35.56%  <- after adaptation')
print('=' * 55)

NEUROREACH MODULE 1 — RESULTS SUMMARY
Model                     Dataset    Accuracy
-------------------------------------------------------
ResNet18 (CNN)            OASIS      99.89%
ResNet18 (CNN)            ADNI       30.37%  <- domain gap
ViT                       OASIS      99.98%
ViT + CORAL               ADNI       35.56%  <- after adaptation


In [17]:
# ============================================================
# Cell 14 — Split ADNI into Train/Val/Test (3 classes)
# ============================================================
# We use ADNI labeled data only for fine-tuning.
# 3 classes: CN=0, MCI=1, AD=2
# Split: 70/15/15

from torch.utils.data import random_split

# Remap labels to 3 classes
GROUP_MAP_3 = {'CN': 0, 'MCI': 1, 'AD': 2}
class_names_3 = ['CN (Normal)', 'MCI', 'AD (Alzheimer\'s)']

subject_to_label_3 = {
    row['Subject']: GROUP_MAP_3[row['Group']]
    for _, row in subject_labels.iterrows()
}

class ADNIDataset3Class(Dataset):
    def __init__(self, subject_to_files, subject_to_label, slices_per_subject=10, transform=None):
        self.transform = transform
        self.samples = []
        for subj, files in subject_to_files.items():
            if subj not in subject_to_label:
                continue
            label = subject_to_label[subj]
            n = len(files)
            start = int(0.20 * n)
            end   = int(0.80 * n)
            usable = files[start:end]
            indices = np.linspace(0, len(usable) - 1, slices_per_subject, dtype=int)
            for i in indices:
                self.samples.append((usable[i], label))
        print(f'Total samples: {len(self.samples)}')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        dcm = pydicom.dcmread(path)
        img = dcm.pixel_array.astype(np.float32)
        img = (img - img.min()) / (img.max() - img.min() + 1e-8)
        img = np.stack([img, img, img], axis=0)
        img = torch.tensor(img, dtype=torch.float32)
        if self.transform:
            img = self.transform(img)
        return img, label

# Build full ADNI dataset
adni_full = ADNIDataset3Class(
    subject_to_files,
    subject_to_label_3,
    slices_per_subject=10,
    transform=adni_transforms
)

# Split
total_adni = len(adni_full)
train_adni = int(0.70 * total_adni)
val_adni   = int(0.15 * total_adni)
test_adni  = total_adni - train_adni - val_adni

adni_train, adni_val, adni_test = random_split(
    adni_full,
    [train_adni, val_adni, test_adni],
    generator=torch.Generator().manual_seed(42)
)

adni_train_loader = DataLoader(adni_train, batch_size=16, shuffle=True)
adni_val_loader   = DataLoader(adni_val,   batch_size=16, shuffle=False)
adni_test_loader  = DataLoader(adni_test,  batch_size=16, shuffle=False)

print(f'Train: {train_adni} | Val: {val_adni} | Test: {test_adni}')

Total samples: 270
Train: 189 | Val: 40 | Test: 41


In [20]:
# ============================================================
# Cell 15 — 5-Fold Cross Validation: ViT Fine-tuning on ADNI
# ============================================================

import timm
from torch.utils.data import SubsetRandomSampler

!kaggle datasets download -d mohinarustamova/neuroreach-vit-weights -p /content/vit --unzip
VIT_PATH = '/content/vit/vit_finetune.pth'

def build_vit_3class():
    """Build fresh ViT with frozen backbone and 3-class head."""
    vit = timm.create_model('vit_base_patch16_224', pretrained=False)
    vit.head = nn.Linear(vit.head.in_features, 4)
    vit.load_state_dict(torch.load(VIT_PATH, map_location=device))
    vit.head = nn.Linear(vit.head.in_features, 3)
    # Freeze all except head
    for param in vit.parameters():
        param.requires_grad = False
    for param in vit.head.parameters():
        param.requires_grad = True
    return vit.to(device)

# Build full ADNI dataset (all 270 samples)
adni_full = ADNIDataset3Class(
    subject_to_files,
    subject_to_label_3,
    slices_per_subject=10,
    transform=adni_transforms
)

from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)
indices = list(range(len(adni_full)))

fold_accuracies = []
fold_reports = []

for fold, (train_idx, test_idx) in enumerate(kf.split(indices)):
    print(f'\n--- Fold {fold+1}/5 ---')

    train_loader_fold = DataLoader(adni_full, batch_size=16,
                                   sampler=SubsetRandomSampler(train_idx))
    test_loader_fold  = DataLoader(adni_full, batch_size=16,
                                   sampler=SubsetRandomSampler(test_idx))

    # Fresh model for each fold
    vit_fold = build_vit_3class()
    optimizer_fold = optim.Adam(vit_fold.parameters(), lr=1e-3)
    criterion_fold = nn.CrossEntropyLoss()

    # Train for 20 epochs
    for epoch in range(20):
        vit_fold.train()
        correct, total = 0, 0
        for imgs, lbls in train_loader_fold:
            imgs, lbls = imgs.to(device), lbls.to(device)
            optimizer_fold.zero_grad()
            outputs = vit_fold(imgs)
            loss = criterion_fold(outputs, lbls)
            loss.backward()
            optimizer_fold.step()
            preds = outputs.argmax(dim=1)
            correct += (preds == lbls).sum().item()
            total += lbls.size(0)
        if (epoch + 1) % 5 == 0:
            print(f'  Epoch {epoch+1}/20 Train Acc: {correct/total*100:.2f}%')

    # Evaluate on test fold
    vit_fold.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, lbls in test_loader_fold:
            imgs = imgs.to(device)
            outputs = vit_fold(imgs)
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(lbls.numpy())

    acc = np.mean(np.array(all_preds) == np.array(all_labels)) * 100
    fold_accuracies.append(acc)
    print(f'Fold {fold+1} Test Accuracy: {acc:.2f}%')

print(f'\n=== 5-Fold CV Results ===')
print(f'Per fold: {[f"{a:.2f}%" for a in fold_accuracies]}')
print(f'Mean Accuracy : {np.mean(fold_accuracies):.2f}%')
print(f'Std Deviation : {np.std(fold_accuracies):.2f}%')

Dataset URL: https://www.kaggle.com/datasets/mohinarustamova/neuroreach-vit-weights
License(s): unknown
100% 304M/304M [00:02<00:00, 112MB/s]

Total samples: 270

--- Fold 1/5 ---
  Epoch 5/20 Train Acc: 78.70%
  Epoch 10/20 Train Acc: 97.22%
  Epoch 15/20 Train Acc: 98.61%
  Epoch 20/20 Train Acc: 99.54%
Fold 1 Test Accuracy: 48.15%

--- Fold 2/5 ---
  Epoch 5/20 Train Acc: 78.24%
  Epoch 10/20 Train Acc: 95.37%
  Epoch 15/20 Train Acc: 99.07%
  Epoch 20/20 Train Acc: 99.54%
Fold 2 Test Accuracy: 48.15%

--- Fold 3/5 ---
  Epoch 5/20 Train Acc: 82.41%
  Epoch 10/20 Train Acc: 97.22%
  Epoch 15/20 Train Acc: 99.54%
  Epoch 20/20 Train Acc: 99.54%
Fold 3 Test Accuracy: 51.85%

--- Fold 4/5 ---
  Epoch 5/20 Train Acc: 83.33%
  Epoch 10/20 Train Acc: 94.91%
  Epoch 15/20 Train Acc: 98.15%
  Epoch 20/20 Train Acc: 99.07%
Fold 4 Test Accuracy: 50.00%

--- Fold 5/5 ---
  Epoch 5/20 Train Acc: 76.39%
  Epoch 10/20 Train Acc: 94.44%
  Epoch 15/20 Train Acc: 99.07%
  Epoch 20/20 Train Acc: 100.

In [21]:
# ============================================================
# Cell 16 — Improved Fine-tuning: More Slices + Unfreeze Last 2 Blocks
# ============================================================
# Changes from Cell 15:
#   1. slices_per_subject = 20 (doubles dataset to 540 samples)
#   2. Unfreeze last 2 transformer blocks + head (~14M params)
#   3. Lower LR = 1e-4 (more params need smaller LR)

# Build larger ADNI dataset (20 slices per subject)
adni_full_20 = ADNIDataset3Class(
    subject_to_files,
    subject_to_label_3,
    slices_per_subject=20,
    transform=adni_transforms
)
print(f'Dataset size with 20 slices: {len(adni_full_20)}')

def build_vit_partial_unfreeze():
    """ViT with last 2 blocks + head unfrozen."""
    vit = timm.create_model('vit_base_patch16_224', pretrained=False)
    vit.head = nn.Linear(vit.head.in_features, 4)
    vit.load_state_dict(torch.load(VIT_PATH, map_location=device))
    vit.head = nn.Linear(vit.head.in_features, 3)
    # Freeze all
    for param in vit.parameters():
        param.requires_grad = False
    # Unfreeze last 2 transformer blocks
    for block in vit.blocks[-2:]:
        for param in block.parameters():
            param.requires_grad = True
    # Unfreeze head
    for param in vit.head.parameters():
        param.requires_grad = True
    trainable = sum(p.numel() for p in vit.parameters() if p.requires_grad)
    print(f'Trainable params: {trainable:,}')
    return vit.to(device)

# 5-Fold CV with improved setup
kf2 = KFold(n_splits=5, shuffle=True, random_state=42)
indices2 = list(range(len(adni_full_20)))
fold_accuracies_2 = []

for fold, (train_idx, test_idx) in enumerate(kf2.split(indices2)):
    print(f'\n--- Fold {fold+1}/5 ---')

    train_loader_fold = DataLoader(adni_full_20, batch_size=16,
                                   sampler=SubsetRandomSampler(train_idx))
    test_loader_fold  = DataLoader(adni_full_20, batch_size=16,
                                   sampler=SubsetRandomSampler(test_idx))

    vit_fold = build_vit_partial_unfreeze()
    optimizer_fold = optim.Adam(vit_fold.parameters(), lr=1e-4)
    criterion_fold = nn.CrossEntropyLoss()

    for epoch in range(20):
        vit_fold.train()
        correct, total = 0, 0
        for imgs, lbls in train_loader_fold:
            imgs, lbls = imgs.to(device), lbls.to(device)
            optimizer_fold.zero_grad()
            outputs = vit_fold(imgs)
            loss = criterion_fold(outputs, lbls)
            loss.backward()
            optimizer_fold.step()
            preds = outputs.argmax(dim=1)
            correct += (preds == lbls).sum().item()
            total += lbls.size(0)
        if (epoch + 1) % 5 == 0:
            print(f'  Epoch {epoch+1}/20 Train Acc: {correct/total*100:.2f}%')

    vit_fold.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, lbls in test_loader_fold:
            imgs = imgs.to(device)
            outputs = vit_fold(imgs)
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(lbls.numpy())

    acc = np.mean(np.array(all_preds) == np.array(all_labels)) * 100
    fold_accuracies_2.append(acc)
    print(f'Fold {fold+1} Test Accuracy: {acc:.2f}%')

print(f'\n=== Improved 5-Fold CV Results ===')
print(f'Per fold: {[f"{a:.2f}%" for a in fold_accuracies_2]}')
print(f'Mean Accuracy : {np.mean(fold_accuracies_2):.2f}%')
print(f'Std Deviation : {np.std(fold_accuracies_2):.2f}%')
print(f'\nImprovement over Cell 15: {np.mean(fold_accuracies_2) - 50.00:.2f}%')

Total samples: 540
Dataset size with 20 slices: 540

--- Fold 1/5 ---
Trainable params: 14,178,051
  Epoch 5/20 Train Acc: 98.84%
  Epoch 10/20 Train Acc: 100.00%
  Epoch 15/20 Train Acc: 100.00%
  Epoch 20/20 Train Acc: 100.00%
Fold 1 Test Accuracy: 63.89%

--- Fold 2/5 ---
Trainable params: 14,178,051
  Epoch 5/20 Train Acc: 100.00%
  Epoch 10/20 Train Acc: 100.00%
  Epoch 15/20 Train Acc: 100.00%
  Epoch 20/20 Train Acc: 100.00%
Fold 2 Test Accuracy: 67.59%

--- Fold 3/5 ---
Trainable params: 14,178,051
  Epoch 5/20 Train Acc: 100.00%
  Epoch 10/20 Train Acc: 100.00%
  Epoch 15/20 Train Acc: 100.00%
  Epoch 20/20 Train Acc: 100.00%
Fold 3 Test Accuracy: 76.85%

--- Fold 4/5 ---
Trainable params: 14,178,051
  Epoch 5/20 Train Acc: 100.00%
  Epoch 10/20 Train Acc: 100.00%
  Epoch 15/20 Train Acc: 100.00%
  Epoch 20/20 Train Acc: 100.00%
Fold 4 Test Accuracy: 69.44%

--- Fold 5/5 ---
Trainable params: 14,178,051
  Epoch 5/20 Train Acc: 100.00%
  Epoch 10/20 Train Acc: 100.00%
  Epoch 1

In [23]:
# ============================================================
# Cell 17 — ViT + CORAL + Fine-tuned on ADNI (Best Model)
# ============================================================

!kaggle datasets download -d mohinarustamova/neuroreach-coral-weights -p /content/coral --unzip

import os
print("Files in /content/coral:")
for f in os.listdir('/content/coral'):
    print(f)

CORAL_PATH = '/content/coral/vit_coral_adapted.pth'

def build_vit_from_coral():
    """Load CORAL-adapted ViT, replace head with 3-class head."""
    vit = timm.create_model('vit_base_patch16_224', pretrained=False)
    vit.head = nn.Linear(vit.head.in_features, 4)
    coral_weights = torch.load(CORAL_PATH, map_location=device)
    vit.load_state_dict(coral_weights, strict=False)
    vit.head = nn.Linear(vit.head.in_features, 3)
    # Freeze all, unfreeze last 2 blocks + head
    for param in vit.parameters():
        param.requires_grad = False
    for block in vit.blocks[-2:]:
        for param in block.parameters():
            param.requires_grad = True
    for param in vit.head.parameters():
        param.requires_grad = True
    trainable = sum(p.numel() for p in vit.parameters() if p.requires_grad)
    print(f'Trainable params: {trainable:,}')
    return vit.to(device)

# 5-Fold CV with CORAL-adapted weights
kf3 = KFold(n_splits=5, shuffle=True, random_state=42)
indices3 = list(range(len(adni_full_20)))
fold_accuracies_3 = []

for fold, (train_idx, test_idx) in enumerate(kf3.split(indices3)):
    print(f'\n--- Fold {fold+1}/5 ---')

    train_loader_fold = DataLoader(adni_full_20, batch_size=16,
                                   sampler=SubsetRandomSampler(train_idx))
    test_loader_fold  = DataLoader(adni_full_20, batch_size=16,
                                   sampler=SubsetRandomSampler(test_idx))

    vit_fold = build_vit_from_coral()
    optimizer_fold = optim.Adam(vit_fold.parameters(), lr=1e-4)
    criterion_fold = nn.CrossEntropyLoss()

    for epoch in range(20):
        vit_fold.train()
        correct, total = 0, 0
        for imgs, lbls in train_loader_fold:
            imgs, lbls = imgs.to(device), lbls.to(device)
            optimizer_fold.zero_grad()
            outputs = vit_fold(imgs)
            loss = criterion_fold(outputs, lbls)
            loss.backward()
            optimizer_fold.step()
            preds = outputs.argmax(dim=1)
            correct += (preds == lbls).sum().item()
            total += lbls.size(0)
        if (epoch + 1) % 5 == 0:
            print(f'  Epoch {epoch+1}/20 Train Acc: {correct/total*100:.2f}%')

    vit_fold.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, lbls in test_loader_fold:
            imgs = imgs.to(device)
            outputs = vit_fold(imgs)
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(lbls.numpy())

    acc = np.mean(np.array(all_preds) == np.array(all_labels)) * 100
    fold_accuracies_3.append(acc)
    print(f'Fold {fold+1} Test Accuracy: {acc:.2f}%')

print(f'\n=== ViT + CORAL + Fine-tuned Results ===')
print(f'Per fold: {[f"{a:.2f}%" for a in fold_accuracies_3]}')
print(f'Mean Accuracy : {np.mean(fold_accuracies_3):.2f}%')
print(f'Std Deviation : {np.std(fold_accuracies_3):.2f}%')
print(f'\nVs ViT fine-tuned only: {np.mean(fold_accuracies_3) - 70.56:.2f}% difference')

Dataset URL: https://www.kaggle.com/datasets/mohinarustamova/neuroreach-coral-weights
License(s): unknown
100% 609M/609M [00:08<00:00, 75.2MB/s]

Files in /content/coral:
vit_coral_adapted.pth
coral_loss_curves.png
adni_confusion_matrix.png
vit_coral_full.pth
tsne_coral.png

--- Fold 1/5 ---
Trainable params: 14,178,051
  Epoch 5/20 Train Acc: 41.20%
  Epoch 10/20 Train Acc: 54.86%
  Epoch 15/20 Train Acc: 81.48%
  Epoch 20/20 Train Acc: 96.76%
Fold 1 Test Accuracy: 45.37%

--- Fold 2/5 ---
Trainable params: 14,178,051
  Epoch 5/20 Train Acc: 38.19%
  Epoch 10/20 Train Acc: 65.05%
  Epoch 15/20 Train Acc: 83.10%
  Epoch 20/20 Train Acc: 92.82%
Fold 2 Test Accuracy: 53.70%

--- Fold 3/5 ---
Trainable params: 14,178,051
  Epoch 5/20 Train Acc: 38.43%
  Epoch 10/20 Train Acc: 45.14%
  Epoch 15/20 Train Acc: 75.69%
  Epoch 20/20 Train Acc: 89.81%
Fold 3 Test Accuracy: 58.33%

--- Fold 4/5 ---
Trainable params: 14,178,051
  Epoch 5/20 Train Acc: 38.43%
  Epoch 10/20 Train Acc: 58.33%
  Epoc

In [24]:
# ============================================================
# Cell 18 — ViT + CORAL + Fine-tuned (Last 4 Blocks)
# ============================================================
# Previous attempt with 2 blocks gave 51.67%
# Unfreezing 4 blocks gives more capacity to recover
# from CORAL feature distortion

def build_vit_from_coral_4blocks():
    """Load CORAL-adapted ViT, unfreeze last 4 blocks + head."""
    vit = timm.create_model('vit_base_patch16_224', pretrained=False)
    vit.head = nn.Linear(vit.head.in_features, 4)
    coral_weights = torch.load(CORAL_PATH, map_location=device)
    vit.load_state_dict(coral_weights, strict=False)
    vit.head = nn.Linear(vit.head.in_features, 3)
    # Freeze all
    for param in vit.parameters():
        param.requires_grad = False
    # Unfreeze last 4 blocks
    for block in vit.blocks[-4:]:
        for param in block.parameters():
            param.requires_grad = True
    # Unfreeze head
    for param in vit.head.parameters():
        param.requires_grad = True
    trainable = sum(p.numel() for p in vit.parameters() if p.requires_grad)
    print(f'Trainable params: {trainable:,}')
    return vit.to(device)

# 5-Fold CV
kf4 = KFold(n_splits=5, shuffle=True, random_state=42)
indices4 = list(range(len(adni_full_20)))
fold_accuracies_4 = []

for fold, (train_idx, test_idx) in enumerate(kf4.split(indices4)):
    print(f'\n--- Fold {fold+1}/5 ---')

    train_loader_fold = DataLoader(adni_full_20, batch_size=16,
                                   sampler=SubsetRandomSampler(train_idx))
    test_loader_fold  = DataLoader(adni_full_20, batch_size=16,
                                   sampler=SubsetRandomSampler(test_idx))

    vit_fold = build_vit_from_coral_4blocks()
    optimizer_fold = optim.Adam(vit_fold.parameters(), lr=1e-4)
    criterion_fold = nn.CrossEntropyLoss()

    for epoch in range(20):
        vit_fold.train()
        correct, total = 0, 0
        for imgs, lbls in train_loader_fold:
            imgs, lbls = imgs.to(device), lbls.to(device)
            optimizer_fold.zero_grad()
            outputs = vit_fold(imgs)
            loss = criterion_fold(outputs, lbls)
            loss.backward()
            optimizer_fold.step()
            preds = outputs.argmax(dim=1)
            correct += (preds == lbls).sum().item()
            total += lbls.size(0)
        if (epoch + 1) % 5 == 0:
            print(f'  Epoch {epoch+1}/20 Train Acc: {correct/total*100:.2f}%')

    vit_fold.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, lbls in test_loader_fold:
            imgs = imgs.to(device)
            outputs = vit_fold(imgs)
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(lbls.numpy())

    acc = np.mean(np.array(all_preds) == np.array(all_labels)) * 100
    fold_accuracies_4.append(acc)
    print(f'Fold {fold+1} Test Accuracy: {acc:.2f}%')

print(f'\n=== ViT + CORAL + Fine-tuned (4 blocks) Results ===')
print(f'Per fold: {[f"{a:.2f}%" for a in fold_accuracies_4]}')
print(f'Mean Accuracy : {np.mean(fold_accuracies_4):.2f}%')
print(f'Std Deviation : {np.std(fold_accuracies_4):.2f}%')
print(f'\nVs ViT fine-tuned only (70.56%): {np.mean(fold_accuracies_4) - 70.56:.2f}%')
print(f'Vs CORAL + 2 blocks (51.67%): {np.mean(fold_accuracies_4) - 51.67:.2f}%')


--- Fold 1/5 ---
Trainable params: 28,353,795
  Epoch 5/20 Train Acc: 40.51%
  Epoch 10/20 Train Acc: 46.30%
  Epoch 15/20 Train Acc: 69.44%
  Epoch 20/20 Train Acc: 88.43%
Fold 1 Test Accuracy: 47.22%

--- Fold 2/5 ---
Trainable params: 28,353,795
  Epoch 5/20 Train Acc: 35.88%
  Epoch 10/20 Train Acc: 46.06%
  Epoch 15/20 Train Acc: 75.46%
  Epoch 20/20 Train Acc: 86.81%
Fold 2 Test Accuracy: 55.56%

--- Fold 3/5 ---
Trainable params: 28,353,795
  Epoch 5/20 Train Acc: 39.58%
  Epoch 10/20 Train Acc: 40.05%
  Epoch 15/20 Train Acc: 52.31%
  Epoch 20/20 Train Acc: 78.47%
Fold 3 Test Accuracy: 50.93%

--- Fold 4/5 ---
Trainable params: 28,353,795
  Epoch 5/20 Train Acc: 37.27%
  Epoch 10/20 Train Acc: 43.29%
  Epoch 15/20 Train Acc: 71.53%
  Epoch 20/20 Train Acc: 90.97%
Fold 4 Test Accuracy: 47.22%

--- Fold 5/5 ---
Trainable params: 28,353,795
  Epoch 5/20 Train Acc: 39.35%
  Epoch 10/20 Train Acc: 39.58%
  Epoch 15/20 Train Acc: 52.08%
  Epoch 20/20 Train Acc: 76.39%
Fold 5 Test Ac

In [25]:
# ============================================================
# Cell 19 — Save All Results to Google Drive
# ============================================================

import json

results = {
    'resnet18_oasis'              : 99.89,
    'vit_oasis'                   : 99.98,
    'resnet18_adni_zeroshot'      : 30.37,
    'vit_coral_adni'              : 35.56,
    'vit_finetuned_adni_mean'     : 70.56,
    'vit_finetuned_adni_std'      : 4.77,
    'vit_finetuned_adni_folds'    : fold_accuracies_2,
    'vit_coral_finetuned_2blocks' : 51.67,
    'vit_coral_finetuned_4blocks' : 49.44,
}

# Save results JSON
with open('/content/drive/MyDrive/NeuroReach/module1_results.json', 'w') as f:
    json.dump(results, f, indent=4)
print('Results saved to Google Drive.')

# Print final summary
print('\n' + '='*55)
print('NEUROREACH MODULE 1 — FINAL RESULTS')
print('='*55)
print(f'{"Model":<35} {"Accuracy"}')
print('-'*55)
print(f'{"ResNet18 on OASIS":<35} 99.89%')
print(f'{"ViT on OASIS":<35} 99.98%')
print(f'{"ResNet18 on ADNI (zero-shot)":<35} 30.37%')
print(f'{"ViT + CORAL on ADNI":<35} 35.56%')
print(f'{"ViT fine-tuned on ADNI":<35} 70.56% ± 4.77%')
print(f'{"ViT + CORAL + fine-tuned":<35} 51.67%')
print('='*55)
print('\nAll results saved to Google Drive/NeuroReach/')

Results saved to Google Drive.

NEUROREACH MODULE 1 — FINAL RESULTS
Model                               Accuracy
-------------------------------------------------------
ResNet18 on OASIS                   99.89%
ViT on OASIS                        99.98%
ResNet18 on ADNI (zero-shot)        30.37%
ViT + CORAL on ADNI                 35.56%
ViT fine-tuned on ADNI              70.56% ± 4.77%
ViT + CORAL + fine-tuned            51.67%

All results saved to Google Drive/NeuroReach/
